# **Goal: Create a code explanation for each cell as text below it.**

**Creating a hybrid search system using**
* Embeddings for semantic search (sentence_transformers)
* BM25 for keyword ranking (Sparse retrieval)
* FAISS as a index.









In [1]:
!pip install sentence-transformers

In [2]:
!pip install rank_bm25

In [3]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 30.8 MB/s eta 0:00:00


In [4]:
import sentence_transformers

In [5]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss


Install and Import all required libraries
- sentence_transformers for generating embeddings
- rank_bm25 for keyword-based ranking
- numpy for numerical operations
- faiss for efficient similarity search indexing.


In [6]:
documents = [
    "Artificial Intelligence is changing the world.",
    "Machine Learning is a subset of AI.",
    "Deep Learning is a subset of Machine Learning.",
    "Natural Language Processing involves understanding text.",
    "Computer Vision allows machines to see and understand.",
    "AI includes areas like NLP and Computer Vision.",
    "The Pyramids of Giza are architectural marvels.",
    "Mozart was a prolific composer during the classical era.",
    "Mount Everest is the tallest mountain on Earth.",
    "The Nile is one of the world's longest rivers.",
    "Van Gogh's Starry Night is a popular piece of art.",
    "Basketball is a sport played with a round ball and two teams."
]

In [7]:
query = "Tell me about AI in text and vision."

Create a sample list of documents covering diverse topics like AI, nature, and sports. This is our corpus that will be searched using the hybrid approach.


In [8]:
tokenized_corpus = [doc.split(" ") for doc in documents]


Tokenize each document by splitting it into words. This tokenized format is required by BM25 for keyword-based ranking calculations.


In [9]:
bm25 = BM25Okapi(tokenized_corpus)


Initialize BM25Okapi with the tokenized corpus. BM25 is a probabilistic ranking function that scores documents based on keyword relevance.


In [ ]:
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

In [11]:
document_embeddings = model.encode(documents)

Load a pre-trained sentence transformer model (paraphrase-MiniLM-L6-v2) and generate dense vector embeddings for every document. These embeddings capture semantic meaning.


In [12]:
index = faiss.IndexFlatL2(document_embeddings.shape[1])

Create a FAISS index with L2 (Euclidean) distance metric. This index will enable fast similarity search over the document embeddings.


In [13]:
index.add(np.array(document_embeddings).astype('float32'))


Add all document embeddings to the FAISS index. The index now contains all documents and is ready for similarity queries.


In [14]:
top_n =10

Set the number of top documents to retrieve in our hybrid search pipeline.


In [15]:
bm25_scores = bm25.get_scores(query.split(" "))

Calculate BM25 scores for the query. This gives keyword-based relevance scores for each document in the corpus.


In [16]:
top_docs_indices = np.argsort(bm25_scores)[-top_n:]

Get the indices of the top 10 documents based on BM25 scores. This is the first stage of our hybrid search, filtering by keyword relevance.


In [17]:
top_docs_embeddings = [document_embeddings[i] for i in top_docs_indices]

Extract the embeddings of only the top BM25-ranked documents. We now have a smaller, relevant subset to perform dense semantic search on.

In [18]:
query_embedding = model.encode([query])

Generate the embedding for the user's query using the same transformer model. This allows semantic comparison with document embeddings.

In [19]:
sub_index = faiss.IndexFlatL2(top_docs_embeddings[0].shape[0])

Create a new FAISS index containing only the top BM25-ranked documents. This enables fast dense semantic search on the filtered set.


In [20]:
sub_index.add(np.array(top_docs_embeddings).astype('float32'))

Add the BM25-filtered embeddings to the sub-index. Now we can search semantically within the keyword-relevant subset.


In [24]:
_,sub_dense_ranked_indices = sub_index.search(np.array(query_embedding).astype('float32'), top_n)


Search for semantically similar documents in the BM25-filtered subset. This is the second stage of our hybrid approach, ranking by semantic relevance.


In [25]:
sub_dense_ranked_indices


array([[9, 8, 1, 0, 6, 7, 2, 4, 3, 5]])

Display the indices from the dense search within the BM25-filtered subset.

In [26]:
final_ranked_indices = [top_docs_indices[i] for i in sub_dense_ranked_indices[0]]

Map the indices from the sub-index back to the original document indices. This gives us the final ranking from our hybrid search system.


In [27]:
ranked_docs = [documents[i] for i in final_ranked_indices]

Retrieve the actual document texts in the final ranked order. These are the top results from our hybrid BM25 + FAISS search system.

In [28]:
ranked_docs

['AI includes areas like NLP and Computer Vision.',
 'Computer Vision allows machines to see and understand.',
 'Natural Language Processing involves understanding text.',
 'Deep Learning is a subset of Machine Learning.',
 "Van Gogh's Starry Night is a popular piece of art.",
 'Basketball is a sport played with a round ball and two teams.',
 'Mozart was a prolific composer during the classical era.',
 "The Nile is one of the world's longest rivers.",
 'The Pyramids of Giza are architectural marvels.',
 'Mount Everest is the tallest mountain on Earth.']

# Provide a brief description of the process this code implements.

#### Answer
This hybrid search system combines BM25, sparse retrieval, for keyword matching with FAISS, dense retrieval, for semantic similarity.

In first Stage BM25 scores documents based on how well their keywords match the query, quickly narrowing down to the most relevant documents.
Second Stage is Semantic Ranking, FAISS then re-ranks these filtered documents by measuring semantic similarity, understanding the deeper meaning beyond just keywords.

This approach is powerful because, by combining both methods, we get the precision of keyword matching with the intelligence of semantic understanding. This means we retrieve documents that are both relevant to what we're searching for, keywords matter, and truly aligned with what we mean, semantics matter.

The real-world benefit of this is instead of missing relevant documents because they use different words, or getting irrelevant results that happen to match keywords, we get more accurate and meaningful search results.
